# TTD Databricks SDK - Example Notebook

This notebook demonstrates how to use the `ttd-databricks` SDK to submit audience data
to The Trade Desk's Data API from a Databricks environment.

## Prerequisites
- Package installed: `%pip install ttd-databricks ttd-data`
- A valid TTD API token (TTD-Auth)
- A valid TTD Data Provider ID
- A valid TTD Advertiser ID (for advertiser endpoint)

In [ ]:
%pip install ttd-databricks ttd-data

## Step 1: Configure credentials

In production, retrieve secrets from Databricks Secrets:
```python
api_token = dbutils.secrets.get(scope="ttd", key="api-token")
```

In [ ]:
# Replace with your actual values
API_TOKEN = "<your-ttd-auth-token>"
DATA_PROVIDER_ID = "<your-data-provider-id>"
ADVERTISER_ID = "<your-advertiser-id>"

## Step 2: Create the client

Use `from_params()` to create the client from your credentials.
The SparkSession is auto-detected from the Databricks runtime.

In [ ]:
from ttd_databricks_python.ttd_databricks import (
    TtdDatabricksClient,
    AdvertiserContext,
    TTDEndpoint,
    get_ttd_input_schema,
    TTDSchemaValidationError,
    TTDApiError,
)

client = TtdDatabricksClient.from_params(api_token=API_TOKEN)
print("Client ready.")

## Step 3: Create a context

The context holds endpoint-specific config: which advertiser and data provider
the data belongs to.

In [ ]:
context = AdvertiserContext(
    data_provider_id=DATA_PROVIDER_ID,
    advertiser_id=ADVERTISER_ID,
)
print(f"Context: {context}")

## Step 4: Inspect the required input schema

Use `get_ttd_input_schema()` to see which columns your DataFrame must contain.

In [ ]:
input_schema = get_ttd_input_schema(TTDEndpoint.ADVERTISER)
print("Required input schema:")
input_schema.printTreeString()

## Step 5: Prepare your input DataFrame

Your DataFrame must contain at minimum the mandatory columns.
Extra columns are allowed and will be preserved in the output.

In [ ]:
# Example: create a small sample DataFrame
# In practice, read from a Delta table or other data source
sample_data = [
    ("tdid-001", "seg-001"),
    ("tdid-002", "seg-002"),
    ("tdid-003", "seg-001"),
]
input_df = spark.createDataFrame(sample_data, schema=input_schema)
display(input_df)

## Step 6: Submit data via push_data (ad hoc mode)

`push_data` will:
1. Validate your DataFrame has the required columns
2. Send data in batches (default: 10 rows per request)
3. Return a new DataFrame with all original columns plus status columns:
   `success`, `error_code`, `error_message`, `processed_timestamp`

In [ ]:
try:
    result_df = client.push_data(
        df=input_df,
        context=context,
        batch_size=10,
    )
    display(result_df)
except TTDSchemaValidationError as e:
    print(f"Schema validation failed: {e}")
except TTDApiError as e:
    print(f"API call failed (HTTP {e.status_code}): {e}")

## Step 7: Inspect results

In [ ]:
from pyspark.sql.functions import col

total = result_df.count()
succeeded = result_df.filter(col("success") == True).count()
failed = total - succeeded

print(f"Total rows:  {total}")
print(f"Succeeded:   {succeeded}")
print(f"Failed:      {failed}")

if failed > 0:
    print("\nFailed rows:")
    display(result_df.filter(col("success") == False))

## Optional: Table setup for batch processing mode

Use the table setup utilities to create Delta tables with the correct schema,
then use `batch_process()` for ongoing incremental processing.

In [ ]:
# Create tables with default names and managed storage
input_table = client.setup_input_table(endpoint=TTDEndpoint.ADVERTISER)
output_table = client.setup_output_table(endpoint=TTDEndpoint.ADVERTISER)
metadata_table = client.setup_metadata_table()

print(f"Input table:    {input_table}")
print(f"Output table:   {output_table}")
print(f"Metadata table: {metadata_table}")

In [ ]:
# Run batch processing (reads from input_table, writes to output_table)
client.batch_process(
    context=context,
    input_table=input_table,
    output_table=output_table,
    metadata_table=metadata_table,
    process_new_records_only=True,  # incremental: only rows newer than last run
    batch_size=10,
    parallelism=8,
)